# 2026-05-27 SSM 与 local_server 改进记录

这份 notebook 记录了本轮围绕 `SSM` 提取器与 `local_server` 联调所做的改进，重点包括：

- 将 `local_server` 接到 `SSMFactory` 上，形成 `Vue -> SSM -> LLM -> San` 链路
- 将大模型输入从 `Vue + SSM` 调整为 **仅 SSM**
- 修复 `props` 提取被注释污染的问题
- 修复模板中 `@click` 等简写事件指令解析错误
- 修复 AST 路径中的事件名和 `san_event_syntax` 输出


## 1. 本轮问题背景

在对 `data/datasets/components/01_simple/step/vue/step.vue` 做联调时，发现：

1. `props` 名称被错误提取成整段注释 + 对象文本
2. `@click="addSteps"` 被错误保留成事件名 `@click`
3. `san_event_syntax` 被错误生成成 `on-@click="addSteps"`
4. `local_server` 原本更偏向评估模式，和当前“基于 SSM 直接生成 San”目标不完全一致

因此本轮改进同时覆盖了：

- SSM 提取质量
- local_server 生成链路
- 调试与联调文档能力


## 2. local_server 生成链路改造

`local_server/api/evaluation_routes.py` 现在支持两类入口：

- `POST /api/evaluation/extract`：Vue -> SSM
- `POST /api/evaluation/generate`：SSM -> San

其中 `/generate` 的核心改动是：

- 优先使用请求体直接传入的 `ssm`
- 如果没传 `ssm`，才回退为 `vue_source` / `vue_file_path` -> `SSMFactory` 提取
- 给外部大模型的 prompt 改为 **只包含 SSM**，不再拼接 Vue 源码


In [ ]:
# local_server/api/evaluation_routes.py 的核心片段
def _build_generation_prompt(ssm: dict[str, Any], extra_instruction: str = "") -> str:
    ssm_json = json.dumps(ssm, ensure_ascii=False, indent=2)
    return (
        "请仅基于给定 SSM 生成可运行的 San 组件代码。\n"
        "输出要求：\n"
        "1. 直接输出 San 组件代码，默认使用 san.defineComponent。\n"
        "2. 优先遵循 SSM 中的 san_generation_contract、template、script、styles、binding_graph、event_model、style_model。\n"
        "3. 若 SSM 信息不足，可做最小必要假设，但不要编造未出现的复杂业务逻辑。\n"
        "4. 尽量保留组件名、props、data、computed、methods、事件绑定、子组件注册与 scoped 样式语义。\n"
        "5. 除代码外不要输出解释，除非补充要求明确要求。\n\n"
        f"补充要求：{extra_instruction or '保持代码完整，包含 imports、components、template、initData、methods 与必要样式说明。'}\n\n"
        "--- SSM ---\n"
        f"{ssm_json}"
    )

@evaluation_bp.post("/run")
@evaluation_bp.post("/generate")
def run_generation():
    payload = request.get_json(silent=True) or {}
    source_name = "inline.vue"
    extra_instruction = (payload.get("instruction") or "").strip()
    provided_ssm = payload.get("ssm")

    if provided_ssm:
        ssm = provided_ssm
        source_name = payload.get("source_file") or source_name
    else:
        vue_source, source_name = _load_vue_source(payload)
        factory = SSMFactory(use_node_bridge=True)
        ssm = factory.build(vue_source, file_path=source_name, source_file=source_name)

    client = create_llm_client()
    prompt = _build_generation_prompt(ssm, extra_instruction)
    llm_result = client.generate(
        prompt=prompt,
        system_prompt="You are an expert Vue-to-San code generator. Output only San component code.",
    )


### 解释

这段改动把 `local_server` 从“评估 SSM 是否可迁移”的模式，切成了“直接利用 SSM 生成 San”的模式。

这样做的收益是：

- prompt 更短，更聚焦于中间表示
- 强制验证 `SSM` 自身是否足以驱动代码生成
- 可以把提取问题和生成问题拆开定位


## 3. script_extractor：修复 props 被注释污染

问题根因是：在 `_extract_object_properties()` 拆对象属性前，没有先去掉 JS 注释，导致 `props: { // 注释 ... }` 场景里，注释文本被一起参与切分，最终把 prop 名称污染了。


In [ ]:
# SSM/extractors/script_extractor.py 的关键改动
COMMENT_LINE_RE = re.compile(r'(^|\n)\s*//.*?(?=\n|$)')
COMMENT_BLOCK_RE = re.compile(r'/\*.*?\*/', re.DOTALL)

def _strip_js_comments(source: str) -> str:
    source = COMMENT_BLOCK_RE.sub("", source)
    source = COMMENT_LINE_RE.sub(lambda m: m.group(1), source)
    return source

def _extract_object_properties(obj_body: str) -> List[dict]:
    if not obj_body.strip():
        return []

    body = _strip_js_comments(obj_body)
    body = body.strip()
    if body.startswith("{") and body.endswith("}"):
        body = body[1:-1].strip()

    # 后续继续做顶层属性切分

for pp in prop_props:
    pname = pp["key"]
    required = bool(re.search(r'\brequired\s*:\s*true\b', pp["value"]))
    validator = bool(re.search(r'\bvalidator\s*:', pp["value"]))
    type_match = re.search(r'\btype\s*:\s*([A-Za-z_$][\w$]*)', pp["value"])
    default_match = re.search(r'\bdefault\s*:\s*([^,}\n]+)', pp["value"])

    props.append({
        "name": pname,
        "type": type_match.group(1) if type_match else "unknown",
        "required": required,
        "default": default_match.group(1).strip() if default_match else None,
        "validator": validator,
    })


### 解释

这部分修复带来了两个直接收益：

- `props.name` 不再把注释与对象体一起误当成名称
- `props.default` 可以从对象语法里真正提取出来，例如 `0`

对于 `step.vue` 这种写法：

```js
props: {
  // 可选：初始步数
  initialSteps: {
    type: Number,
    default: 0
  }
}
```

修复后就可以正确得到 `initialSteps` 和默认值 `0`。


## 4. template_extractor：修复 `@click` 简写指令解析

问题根因是原来的正则更偏向 `v-on:click` 这种完整写法，对 `@click`、`:class`、`#default` 等简写形式支持不完整。


In [ ]:
# SSM/extractors/template_extractor.py 的关键改动
def _parse(self):
    raw = self.raw
    value = self.raw_value

    # 支持：v-directive:arg.mod1.mod2 / @click.stop / :class / #default
    full_match = re.match(
        r'^(v-[a-z-]+)(?::([a-zA-Z_$][\w.$-]*))?((?:\.[a-zA-Z_$][\w.$-]*)*)$',
        raw
    )
    shorthand_match = re.match(
        r'^([@:#])([a-zA-Z_$][\w.$-]*)?((?:\.[a-zA-Z_$][\w.$-]*)*)$',
        raw
    )

    if full_match:
        prefix = full_match.group(1)
        self.argument = full_match.group(2) or ""
        modifier_text = full_match.group(3) or ""
    elif shorthand_match:
        prefix = shorthand_match.group(1)
        self.argument = shorthand_match.group(2) or ""
        modifier_text = shorthand_match.group(3) or ""
    else:
        self.directive_name = "custom"
        self.argument = raw
        self.expression = value
        return

    self.modifiers = re.findall(r'\.([a-zA-Z_$][\w.$-]*)', modifier_text)
    self.directive_name = self.DIRECTIVE_NAMES.get(prefix, prefix)
    self.expression = value


### 解释

修复后：

- `@click="addSteps"` 可以正确解析为 `directive_name = v-on`、`argument = click`
- `:class` 能正确进入 `v-bind` 分支
- `#default` 能正确识别为插槽简写

这保证了模板层的事件与动态属性，不会因为简写语法而被错误归为 `custom`。


## 5. parse_sfc.cjs：同步修复 AST 路径输出

因为当前生产链路优先走 Node bridge，所以上面在 Python fallback 路径里的修复，也必须同步到 `parse_sfc.cjs`，否则同一个组件在 AST 路径和 fallback 路径下会产生不一致的 `SSM`。


In [ ]:
// SSM/extractors/parse_sfc.cjs 的关键改动
function extractDirectives(props, nodeId, sourceTag, isComponentEvent) {
  const directives = [];
  const eventBindings = [];
  const dynamicAttrs = [];

  for (const prop of props || []) {
    if (prop.type !== 7) continue;
    const directiveName = `v-${prop.name}`;
    const argument = prop.arg && prop.arg.type === 4 ? prop.arg.content : '';
    const expression = prop.exp && prop.exp.content ? prop.exp.content : '';
    const dependencies = extractIdentifiers(expression);
    const modifiers = Object.keys(prop.modifiers || {});
    const normalizedEventName = prop.name === 'on' ? (argument || 'click') : argument;

    const entry = {
      directive_name: directiveName,
      argument: normalizedEventName,
      modifiers,
      expression,
      dependencies,
      san_equivalent: ({
        'v-if': 's-if',
        'v-else-if': 's-else-if',
        'v-else': 's-else',
        'v-for': 's-for',
        'v-on': 'on-event',
        'v-bind': 'attr 绑定',
        'v-model': 'value={= field =}',
        'v-slot': 'slot',
        'v-show': 's-if 或 display 控制',
      })[directiveName] || directiveName,
      migration_note: `${directiveName} 需要迁移到 San`,
    };

    if (prop.name === 'on') {
      const analyzed = analyzeEventExpression(expression);
      eventBindings.push({
        node_id: nodeId,
        element_tag: sourceTag,
        event_name: normalizedEventName,
        modifiers,
        handler_expression: analyzed.handler_expression,
        handler_type: analyzed.handler_type,
        handler_name: analyzed.handler_name,
        arguments: analyzed.arguments,
        is_component_event: !!isComponentEvent,
        san_event_syntax: `on-${normalizedEventName}="${analyzed.handler_name || analyzed.handler_expression}"`,
      });
    }
  }
}

// props default 同步提取
let defaultValue = null;
if (innerName === 'default') defaultValue = nodeToCode(inner.value, scriptContent) || null;
options.props.push({ name: propName, type, required, default: defaultValue, validator });


### 解释

这一步的意义是让两条路径保持一致：

- Python 侧 fallback 提取
- Node/Babel/Vue AST 侧主提取

修复后，两边都会输出：

- `event_name = click`
- `san_event_syntax = on-click="addSteps"`
- `props.default = 0`


## 6. 联调验证结果

对 `step.vue` 做回归验证后，关键结果变为：

- `props[0].name = initialSteps`
- `props[0].default = 0`
- `event_name = click`
- `san_event_syntax = on-click="addSteps"`

说明提取器已经不再把错误的字段继续传给大模型。


In [ ]:
# 回归验证命令示例
from SSM.extractors.factory import SSMFactory

path = '/Users/baidu-yangrunsheng/Desktop/CardMigratorSystem/data/datasets/components/01_simple/step/vue/step.vue'
ssm = SSMFactory(use_node_bridge=True).build_from_file(path)

print('prop names:', [p['name'] for p in ssm['script']['options']['props']])
print('prop defaults:', [p['default'] for p in ssm['script']['options']['props']])
print('event names:', [e['event_name'] for e in ssm['event_model']['dom_events']])
print('san syntax:', [e.get('san_event_syntax') for e in ssm['event_model']['dom_events']])
print('directive args:', [d['argument'] for d in ssm['template']['dom_tree'].get('directives', [])])


## 7. 这次改进的结论

本轮改进把项目推进到了一个更稳定的阶段：

- `local_server` 已能基于 `SSM` 调外部 Qwen API 生成 San
- `SSM` 提取器在简单样例上已经修复了关键错误字段
- `AST` 路径和 fallback 路径的输出更加一致

后续如果继续优化，建议优先关注：

- 更复杂 `props` 语法
- 事件修饰符（如 `.stop`、`.prevent`）
- 生成结果自动校验
